## Example 1: Sign a request with AWS Signature Version 4

In [ ]:
import importlib.metadata

import requests
from aws_requests_auth.aws_auth import AWSRequestsAuth

# Example 1: the auth object adds the SigV4 headers to a prepared request (nothing is sent)
assert importlib.metadata.version("aws-requests-auth") == "0.4.3"
auth = AWSRequestsAuth(aws_access_key="AKIDEXAMPLE", aws_secret_access_key="secret", aws_host="search.example.us-east-1.es.amazonaws.com",
                       aws_region="us-east-1", aws_service="es")
request = requests.Request("GET", "https://search.example.us-east-1.es.amazonaws.com/_cluster/health", auth=auth).prepare()
header = request.headers["Authorization"]
assert header.startswith("AWS4-HMAC-SHA256 Credential=AKIDEXAMPLE/") and "/us-east-1/es/aws4_request" in header, header
assert "x-amz-date" in request.headers and "SignedHeaders=host;x-amz-date" in header
print("Example 1 OK:", header[:90])

## Example 2: A session token is sent and signed

In [ ]:
import requests
from aws_requests_auth.aws_auth import AWSRequestsAuth

# Example 2: temporary credentials add x-amz-security-token, which is part of the signed headers
auth = AWSRequestsAuth(aws_access_key="ASIAEXAMPLE", aws_secret_access_key="secret", aws_token="token-123",
                       aws_host="api.example.com", aws_region="eu-west-1", aws_service="execute-api")
request = requests.Request("POST", "https://api.example.com/items", json={"a": 1}, auth=auth).prepare()
assert request.headers["x-amz-security-token"] == "token-123"
assert "x-amz-security-token" in request.headers["Authorization"], request.headers["Authorization"]
assert "x-amz-content-sha256" in request.headers or "Signature=" in request.headers["Authorization"]
print("Example 2 OK:", request.headers["Authorization"].split("SignedHeaders=")[1].split(",")[0])

## Example 3: The signature is deterministic and scoped to region and service

In [ ]:
import datetime
from unittest import mock

import requests
from aws_requests_auth.aws_auth import AWSRequestsAuth

# Example 3: at a fixed instant the same request signs identically; another region signs differently
def sign(region):
    with mock.patch("aws_requests_auth.aws_auth.datetime") as fake:
        fake.datetime.utcnow.return_value = datetime.datetime(2026, 1, 2, 3, 4, 5)
        auth = AWSRequestsAuth("AK", "SK", "h.example.com", region, "es")
        return requests.Request("GET", "https://h.example.com/p?x=1", auth=auth).prepare().headers


first, again, other = sign("us-east-1"), sign("us-east-1"), sign("eu-west-1")
assert first["x-amz-date"] == "20260102T030405Z" and first["Authorization"] == again["Authorization"]
assert other["Authorization"] != first["Authorization"] and "/eu-west-1/es/aws4_request" in other["Authorization"]
print("Example 3 OK:", first["Authorization"].split("Signature=")[1][:16])

## Example 4: Credentials from botocore's chain (BotoAWSRequestsAuth)

In [ ]:
import os

import requests
from aws_requests_auth.boto_utils import BotoAWSRequestsAuth

# Example 4: BotoAWSRequestsAuth resolves credentials the way boto3 does; here from environment variables
os.environ.update(AWS_ACCESS_KEY_ID="AKIAENVEXAMPLE", AWS_SECRET_ACCESS_KEY="envsecret")
os.environ.pop("AWS_SESSION_TOKEN", None)
auth = BotoAWSRequestsAuth(aws_host="search.example.com", aws_region="us-west-2", aws_service="es")
request = requests.Request("GET", "https://search.example.com/", auth=auth).prepare()
assert "Credential=AKIAENVEXAMPLE/" in request.headers["Authorization"], request.headers["Authorization"]
print("Example 4 OK:", request.headers["Authorization"][:70])

## Example 5: A different body gives a different signature

In [ ]:
import requests
from aws_requests_auth.aws_auth import AWSRequestsAuth

# Example 5: the payload hash is part of the signature, so a tampered body cannot reuse it
auth = AWSRequestsAuth("AK", "SK", "h.example.com", "us-east-1", "es")
first = requests.Request("POST", "https://h.example.com/doc", data=b'{"a":1}', auth=auth).prepare()
second = requests.Request("POST", "https://h.example.com/doc", data=b'{"a":2}', auth=auth).prepare()
if first.headers["x-amz-date"] == second.headers["x-amz-date"]:
    assert first.headers["Authorization"] != second.headers["Authorization"]
print("Example 5 OK: body is signed")